# Sentiment Analysis on Twitter (Sentiment140)

**Portfolio notebook** • February 26, 2026

## Objectives
- Build an end-to-end **sentiment classification** baseline on the **Sentiment140** dataset (1.6M tweets).
- Demonstrate a clean ML workflow: **data loading → EDA → preprocessing → feature engineering → model training → evaluation → interpretation**.
- Produce artifacts suitable for a resume / portfolio: **clear narrative, reproducible code, and interpretable results**.

## Outcomes
By the end of this notebook you will have:
- A **reproducible baseline** using **TF–IDF (unigrams + bigrams)** + **Logistic Regression**.
- An evaluation report (**precision / recall / F1**) and **confusion matrix**.
- A quick look at **misclassified examples** and **top predictive n-grams**.
- A saved sklearn **Pipeline** ready to reuse in another project (e.g., a chatbot intent/sentiment module).


## Dataset: Sentiment140

Sentiment140 is a classic Twitter sentiment dataset.
- **Source**: Kaggle dataset `kazanova/sentiment140`
- **Size**: 1,600,000 tweets (training CSV)
- **Labels**: `0` = negative, `4` = positive

> We'll map labels to **binary**: `0 → 0 (negative)`, `4 → 1 (positive)`.


## Approach (Why this baseline?)

For a first strong baseline on short texts like tweets:
- **TF–IDF** provides a sparse numeric representation of text.
- **Logistic Regression** is fast, strong, and easy to interpret (feature weights).

This combination is a common industry baseline that often performs surprisingly well and is a great starting point before moving to neural methods.


## Notes on runtime (M1 Pro + 16GB)

Training on all 1.6M tweets can take time and memory depending on your vectorizer settings.  
To keep this notebook runnable on most laptops, we include an **optional stratified sample** step. You can set `SAMPLE_SIZE=None` to train on the full dataset.


In [ ]:
# Core libraries
import os
import re
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


In [ ]:
# --- Download the dataset (KaggleHub) ---
# KaggleHub handles downloading and caching datasets without needing a Kaggle API key.
import kagglehub

# Download latest version of the dataset
path = kagglehub.dataset_download("kazanova/sentiment140")
print("Path to dataset files:", path)

# Quick check: list files
os.listdir(path)


## Load the CSV into a DataFrame

In [ ]:
file_path = os.path.join(path, "training.1600000.processed.noemoticon.csv")

# Sentiment140 training CSV has no headers; we assign them.
columns = ["target", "ids", "date", "flag", "user", "text"]

df = pd.read_csv(file_path, encoding="latin-1", names=columns)
df.head()


## Quick sanity checks & EDA

In [ ]:
# Missing values
df.isnull().sum()


In [ ]:
# Distribution of labels before mapping (0 = negative, 4 = positive)
label_counts = df["target"].value_counts().sort_index()
label_counts


In [ ]:
# Plot label distribution
label_counts.plot(kind="bar")
plt.title("Distribution of Sentiment Labels (Original)")
plt.xlabel("Original label")
plt.ylabel("Count")
plt.show()


## Label mapping (0/4 → 0/1)

In [ ]:
# Convert target labels to binary:
# 0 -> 0 (negative)
# 4 -> 1 (positive)
df["target"] = (df["target"] == 4).astype(int)

df["target"].value_counts()


## Optional: stratified sampling for faster iteration

In [ ]:
# If you're iterating locally (e.g., on a laptop), sampling can dramatically reduce runtime.
# Set SAMPLE_SIZE=None to train on the full dataset.
SAMPLE_SIZE = 200_000  # try 200k first; bump up if you have time/memory

if SAMPLE_SIZE is not None and len(df) > SAMPLE_SIZE:
    # Stratified sample: keep label balance
    n_per_class = SAMPLE_SIZE // 2
    df_model = (
        df.groupby("target", group_keys=False)
          .apply(lambda x: x.sample(n=min(n_per_class, len(x)), random_state=RANDOM_SEED))
          .sample(frac=1, random_state=RANDOM_SEED)  # shuffle
          .reset_index(drop=True)
    )
else:
    df_model = df.copy()

df_model.shape, df_model["target"].value_counts()


## Text preprocessing

Tweets contain URLs, mentions, hashtags, and punctuation.  
We do **lightweight cleaning** that works well with TF–IDF:

- lowercase
- replace URLs and @mentions with placeholders (`URL`, `USER`)
- keep hashtag words (remove `#` symbol)
- keep apostrophes (helps with contractions like *don't*)


In [ ]:
# Compile regex once for speed
url_re = re.compile(r"https?://\S+|www\.\S+")
mention_re = re.compile(r"@\w+")
multispace_re = re.compile(r"\s+")

def basic_clean(s: str) -> str:
    """Lightweight tweet cleaning for TF–IDF."""
    s = str(s).lower()
    s = url_re.sub(" URL ", s)
    s = mention_re.sub(" USER ", s)
    s = s.replace("#", "")                      # keep hashtag word, drop '#'
    s = re.sub(r"&amp;", " and ", s)
    s = re.sub(r"[^a-z0-9\s']", " ", s)         # keep words/numbers/apostrophes
    s = multispace_re.sub(" ", s).strip()
    return s

# Preview cleaning on a few examples
sample_rows = df_model.sample(5, random_state=RANDOM_SEED)[["text", "target"]].copy()
sample_rows["cleaned"] = sample_rows["text"].map(basic_clean)
sample_rows


## Train / test split

In [ ]:
X = df_model["text"].map(basic_clean)
y = df_model["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=y
)

len(X_train), len(X_test)


## Modeling: TF–IDF + Logistic Regression (sklearn Pipeline)

Using a `Pipeline` keeps preprocessing and the model tied together:
- prevents accidental data leakage
- makes it easy to save/reuse


In [ ]:
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2),
        min_df=5,
        max_df=0.9
    )),
    ("clf", LogisticRegression(
        max_iter=300,
        solver="liblinear",  # solid default for sparse text baselines
        random_state=RANDOM_SEED
    ))
])

pipeline


In [ ]:
pipeline.fit(X_train, y_train)

pred = pipeline.predict(X_test)
print(classification_report(y_test, pred, digits=4))


In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, pred)
plt.title("Confusion Matrix (Test Set)")
plt.show()


## Quick error analysis (portfolio-friendly)

Looking at mistakes helps you understand limitations:
- sarcasm/irony
- ambiguous wording
- slang and misspellings
- context missing in short tweets


In [ ]:
# Show a few misclassified examples
results = pd.DataFrame({
    "text": X_test.values,
    "y_true": y_test.values,
    "y_pred": pred
})
mistakes = results[results["y_true"] != results["y_pred"]].sample(10, random_state=RANDOM_SEED)
mistakes


## Model interpretation: top predictive n-grams

For linear models, TF–IDF features have weights (coefficients).  
Positive weights push predictions toward **positive sentiment**, negative weights toward **negative sentiment**.


In [ ]:
# Extract feature names and coefficients
tfidf = pipeline.named_steps["tfidf"]
clf = pipeline.named_steps["clf"]

feature_names = np.array(tfidf.get_feature_names_out())
coefs = clf.coef_.ravel()

top_pos_idx = np.argsort(coefs)[-20:][::-1]
top_neg_idx = np.argsort(coefs)[:20]

top_pos = pd.DataFrame({"ngram": feature_names[top_pos_idx], "weight": coefs[top_pos_idx]})
top_neg = pd.DataFrame({"ngram": feature_names[top_neg_idx], "weight": coefs[top_neg_idx]})

top_pos, top_neg


## Save the trained pipeline

This makes your work reusable:
- load the model later for inference
- plug into another app/service


In [ ]:
import joblib

model_path = "sentiment140_tfidf_logreg.joblib"
joblib.dump(pipeline, model_path)

model_path


## Summary & next steps

**What we built**
- A clean baseline for tweet sentiment using **TF–IDF + Logistic Regression**.
- Interpretable features (top positive/negative n-grams).
- A reusable saved **sklearn Pipeline**.

**Next steps (good portfolio extensions)**
- Tune vectorizer + model hyperparameters (e.g., `C`, `min_df`, `max_df`, `ngram_range`).
- Add richer preprocessing (emoji handling, negation handling like "not good").
- Evaluate robustness: performance by tweet length, or by presence of URLs/mentions.
- Try stronger models: linear SVM, or transformer fine-tuning for a higher ceiling.
